In [2]:
import numpy as np
import scipy as sp 
import matplotlib.pyplot as plt

In [9]:
G = 6.67428 * 10**(-11) 
pi = np.pi

## MATHS SECTION ##

def runge_kutta_family(stages, time_vec, stage_vec, A_matrix, n, h, initial_conditions, function):

    t, y = np.append([initial_conditions[0], np.zeros(n - 1)]), np.append([initial_conditions[1], np.zeros(n - 1)])
    k_vec = np.zeros(len(stages))

    for k in range(n - 1): 
        for i in range(n - 1):
            for j in range(i):

                k_vec[i] = function(t[k] + time_vec[i] * h, y[k] + sum(np.multiply(A_matrix[i, j] * k_vec[j]))*h)

        t[k + 1] = t[k] + h
        y[k + 1] = y[k] + h * sum(np.multiply(stage_vec, k_vec))

    return t, y


def linear(x, t):
    return x

def rk4_cts(param, var, h, function):

    k_1 = function(param, var) 
    k_2 = function(param + h/2, var + k_1 * h/2)
    k_3 = function(param + h/2, var + k_2 * h/2)
    k_4 = function(param + h, var + h*k_3)

    k_vec = np.array([k_1, k_2, k_3, k_4])

    return k_vec

def fractured_rk4(k, param, var, h, k_vec):

    param[k + 1] = param[k] + h # n*h = simulation time 
    var[k + 1] = var[k] + h/6 * (k_vec[0] + 2*(k_vec[1] + k_vec[2]) + k_vec[3])

    return var[k + 1], param[k + 1]



## PHYSICS SECTION  ##

def inverse_rad(theta, theta_0, eccent, mu, M, L_0): # solution to Binet's equation, theta=theta_0 determines r_0^-1
    return (G*M*mu/L_0**2) * (1 + eccent * np.cos(theta - theta_0))

def accel(M, r, t):
    return -G*M/r**2

def mod_angular_momentum(mu, r_0, v_0, alpha):
    return mu*r_0*v_0*np.abs(np.sin(alpha))

def eccentricity(r_0, v_0, alpha, mu, M ):

    E_0, L_0 = 1/2*mu*v_0**2 - G*M*mu/r_0, mod_angular_momentum(mu, r_0, v_0, alpha)

    return np.sqrt(1 + 2*E_0*L_0**2 / ((G*M)**2)*mu)


def radial_vel(r, eccent, mu, M, L_0):
    return (L_0**3 *eccent/(G*mu**2 * M)) * np.sqrt( ( 1 + eccent**2)*r**2 - 2*(L_0/(G*M*mu))*r + (L_0**2/(G*M*mu))**2)


# 1. Two Body Problem

In [ ]:
# in MKS everything (for now)
c = 299792458 
M_E = 5.9722 * 10**24 
m_M = 7.34767309 * 10**22 

# Initial Conditions 

h = 5000
n = 20

t_0 = 0
r_0 =  3.844 * 10**8
v_0 = np.sqrt(2*G*M_E/r_0)

# ODE solving 

zeros = np.zeros(n - 1)
param = np.append([t_0], [zeros])
var1 = np.append([r_0], [zeros]) # position
var2 = np.append([v_0], [zeros]) # velocity

for k in range (n - 1):

    r_func = lambda t, r: accel(M_E, r, param)

    k2_vec =  rk4_cts(param[k], var1[k], h, r_func)
    (var2[k + 1], param[k + 1]) = fractured_rk4(k, param, var2, h, k2_vec) # gets velocity out of position

    k1_vec = rk4_cts(param[k], var2[k], h, linear)
    (var1[k + 1], param[k + 1]) = fractured_rk4(k, param, var1, h, k1_vec) # gets position out of velocity 



In [ ]:
red_mass = 10**10
central_mass = 10**20
theta_0 = -pi/3
alpha = pi/6 
init_rad = 10**8
init_vel = 10**4

e = eccentricity(init_rad, init_vel, alpha, red_mass, central_mass)
L_0 = mod_angular_momentum(red_mass, init_rad, init_vel, alpha)

np.float64(7.491439770070908e+25)